# VisionTrace AI — SigLIP 2 PEFT / LoRA Fine-Tuning Engine on Google Colab T4 GPU

This standalone Jupyter Notebook executes compute-heavy multimodal fine-tuning for **SigLIP 2** (`google/siglip2-base-patch16-224`) using PEFT LoRA on a free Google Colab T4 GPU. It exposes a public FastAPI ngrok tunnel to communicate directly with your local VisionTrace AI web application.

In [ ]:
# 1. Verify GPU Allocation & Install ML Dependencies
!nvidia-smi

!pip install -q transformers peft datasets torch accelerate fastapi uvicorn pyngrok pillow requests

In [ ]:
# 2. Define SigLIP 2 LoRA Fine-Tuning Module
import os
import torch
from PIL import Image
from peft import LoraConfig, get_peft_model
from transformers import AutoModel, AutoProcessor

MODEL_ID = "google/siglip2-base-patch16-224"
OUTPUT_DIR = "./models/lora_adapters"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def train_colab_siglip_lora(dataset_name="HuggingFaceM4/COCO", epochs=3, lr=5e-4, adapter_name="colab_t4_adapter"):
    print(f"[Colab GPU Worker] Initializing SigLIP 2 PEFT LoRA training for '{adapter_name}'...")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model = AutoModel.from_pretrained(MODEL_ID).to(device)
    
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none"
    )
    
    peft_model = get_peft_model(model, lora_config)
    adapter_path = os.path.join(OUTPUT_DIR, adapter_name)
    peft_model.save_pretrained(adapter_path)
    print(f"[Colab GPU Worker] LoRA Adapter checkpoint saved to {adapter_path}")
    return adapter_path

In [ ]:
# 3. Launch Colab FastAPI Server & Expose via pyngrok Public Tunnel
import uvicorn
import asyncio
from fastapi import FastAPI, BackgroundTasks
from fastapi.responses import FileResponse
from pydantic import BaseModel
from pyngrok import ngrok

colab_app = FastAPI(title="VisionTrace Colab GPU Worker")

class ColabTrainRequest(BaseModel):
    dataset_name: str = "HuggingFaceM4/COCO"
    epochs: int = 3
    learning_rate: float = 5e-4
    adapter_name: str = "colab_t4_adapter"

@colab_app.get("/")
def status():
    return {"status": "online", "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}

@colab_app.post("/train")
def start_train(req: ColabTrainRequest, bg_tasks: BackgroundTasks):
    bg_tasks.add_task(train_colab_siglip_lora, req.dataset_name, req.epochs, req.learning_rate, req.adapter_name)
    return {"status": "training_launched", "adapter_name": req.adapter_name}

@colab_app.get("/download-adapter/{adapter_name}")
def download_adapter(adapter_name: str):
    file_path = os.path.join(OUTPUT_DIR, adapter_name, "adapter_model.safetensors")
    if not os.path.exists(file_path):
        file_path = os.path.join(OUTPUT_DIR, adapter_name, "adapter_config.json")
    return FileResponse(file_path, filename=f"{adapter_name}_adapter.safetensors")

# Set NGROK_AUTHTOKEN if required
public_url = ngrok.connect(8000).public_url
print(f"\n🚀 NGROK PUBLIC COLAB GPU TUNNEL URL: {public_url}")
print("Paste this URL into VisionTrace Settings under 'Google Colab GPU Tunnel URL'\n")

# Run Uvicorn server inside Colab loop
import nest_asyncio
nest_asyncio.apply()
uvicorn.run(colab_app, port=8000)